# Case NTT DATA - Weirivelton Rodrigues - Medallion Architecture

## Process Flow

1. Authentication  
2. Bronze Layer Read  
3. Bronze Transformation  
4. Silver Layer Write  
5. Silver Layer Read  
6. Incremental Load Processing  
7. Delta MERGE (UPSERT)  
8. Final Validation  

In [0]:

# Library Imports

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

Authentication

In [0]:
# Configuração de autenticação no ADLS Gen2
spark.conf.set(
    "fs.azure.account.key.nttcase.blob.core.windows.net",
    dbutils.secrets.get(scope="ntt-data", key="storage-key")
)

2. Bronze Layer Read  


In [0]:
dbutils.fs.ls(
    "wasbs://bronze@nttcase.blob.core.windows.net/"
)

In [0]:
locations_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(
        "wasbs://bronze@nttcase.blob.core.windows.net/locations/ingestion_date=2026-05-12/locations.csv"
    )

In [0]:
display(locations_df)

3. Bronze Transformation  

suporta carga incremental,
facilita reprocessamento,
melhora particionamento,
reduz custo de leitura,
melhora performance Spark,
e segue padrão enterprise/lakehouse.

In [0]:
from pyspark.sql.functions import expr, lit, to_date, monotonically_increasing_id

data_ref_carga = "2026-05-12"

source_data = locations_df.select(

    expr("""
        CASE
            WHEN location IS NULL
                 OR TRIM(LOWER(location)) = 'null'
                 OR TRIM(location) = ''
            THEN 'N/A'
            ELSE CAST(location AS STRING)
        END AS location
    """),

    expr("""
        CASE
            WHEN iso_code IS NULL
                 OR TRIM(LOWER(iso_code)) = 'null'
                 OR TRIM(iso_code) = ''
            THEN 'N/A'
            ELSE CAST(iso_code AS STRING)
        END AS iso_code
    """),

    expr("""
        CASE
            WHEN vaccines IS NULL
                 OR TRIM(LOWER(vaccines)) = 'null'
                 OR TRIM(vaccines) = ''
            THEN 'N/A'
            ELSE CAST(vaccines AS STRING)
        END AS vaccines
    """),

    expr("""
        CASE
            WHEN last_observation_date IS NULL
            THEN DATE('1900-01-01')
            ELSE CAST(last_observation_date AS DATE)
        END AS last_observation_date
    """),

    expr("""
        CASE
            WHEN source_name IS NULL
                 OR TRIM(LOWER(source_name)) = 'null'
                 OR TRIM(source_name) = ''
            THEN 'N/A'
            ELSE CAST(source_name AS STRING)
        END AS source_name
    """),

    expr("""
        CASE
            WHEN source_website IS NULL
                 OR TRIM(LOWER(source_website)) = 'null'
                 OR TRIM(source_website) = ''
            THEN 'N/A'
            ELSE CAST(source_website AS STRING)
        END AS source_website
    """)

).withColumn(
    "data_ref_carga",
    to_date(lit(data_ref_carga))
)

In [0]:
# Reordena colunas
locations_silver_df = locations_silver_df.select(
    "id",
    "location",
    "iso_code",
    "vaccines",
    "last_observation_date",
    "source_name",
    "source_website",
    "data_ref_carga"
)

In [0]:
display(locations_silver_df)

5 - Silver Layer Write

In [0]:
locations_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("data_ref_carga") \
    .save(
        "wasbs://silver@nttcase.blob.core.windows.net/locations-silver/"
    )

6. Incremental Load Processing 

In [0]:
from delta.tables import DeltaTable

silver_path = "wasbs://silver@nttcase.blob.core.windows.net/locations-silver/"

silver_df = spark.read \
    .format("delta") \
    .load(silver_path)

display(silver_df)

In [0]:
deltaTable = DeltaTable.forPath(
    spark,
    silver_path
)

6 - Delta MERGE (UPSERT)

In [0]:
(
    deltaTable.alias("deltaTable")
    .merge(
        locations_silver_df.alias("source_data"),
        "source_data.id = deltaTable.id"
    )
    .whenMatchedUpdate(set={
        "location": "source_data.location",
        "iso_code": "source_data.iso_code",
        "vaccines": "source_data.vaccines",
        "last_observation_date": "source_data.last_observation_date",
        "source_name": "source_data.source_name",
        "source_website": "source_data.source_website",
        "data_ref_carga": "source_data.data_ref_carga"
    })
    .whenNotMatchedInsert(values={
        "id": "source_data.id",
        "location": "source_data.location",
        "iso_code": "source_data.iso_code",
        "vaccines": "source_data.vaccines",
        "last_observation_date": "source_data.last_observation_date",
        "source_name": "source_data.source_name",
        "source_website": "source_data.source_website",
        "data_ref_carga": "source_data.data_ref_carga"
    })
    .execute()
)

In [0]:
final_df = spark.read \
    .format("delta") \
    .load(silver_path)

In [0]:
final_df.groupBy(
    "data_ref_carga"
).count().show()